# 02 — Official Source Reconciliation

## Kenya Electoral Participation Analytics

### Objective

This notebook reconciles the interim ELOG election dataset created in Notebook 01 against official IEBC source documents.

The notebook will:

- extract official county-level figures where available,
- compare ELOG figures with IEBC figures,
- identify and quantify discrepancies,
- investigate differences in definitions,
- add rejected-ballot and votes-cast information where supported,
- preserve both source values rather than silently overwriting them.

### Official sources used

- IEBC 2013 General Election Data Report
- IEBC 2017 Elections Data Report
- IEBC 2017 Post-Election Evaluation Report
- IEBC 2022 Post-Election Evaluation Report
- Kenya Gazette 2022 Presidential Declaration

The 2022 Kenya Gazette declaration is retained as a legal reference but does not contain the county-level analytical fields required for reconciliation.

In [2]:
# Import Path for working with project folders and files
from pathlib import Path

# Import regular expressions for extracting structured PDF text
import re

# Import pandas for data manipulation and reconciliation
import pandas as pd

# Import PdfReader for extracting text from official PDF sources
from pypdf import PdfReader

# Improve DataFrame display in the notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

In [3]:
# Set the project root as the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent

# Define existing project directories
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REFERENCE_DIR = PROJECT_ROOT / "data" / "raw" / "reference"
DOCS_DIR = PROJECT_ROOT / "documentation"

# Define the ELOG dataset created in Notebook 01
ELOG_FILE = INTERIM_DIR / "elog_county_election_results.csv"

# Define the official reference files used in this notebook
IEBC_2013_FILE = REFERENCE_DIR / "IEBC_2013_General_Election_Results.pdf"
IEBC_2017_DATA_FILE = REFERENCE_DIR / "IEBC_2017_Presidential_Declaration_Results.pdf"
IEBC_2017_EVAL_FILE = REFERENCE_DIR / "IEBC_2017_Post_Election_Evaluation_Report.pdf"
IEBC_2022_EVAL_FILE = REFERENCE_DIR / "IEBC_2022_Post_Election_Evaluation_Report.pdf"
GAZETTE_2022_FILE = REFERENCE_DIR / "Kenya_Gazette_2022_Presidential_Declaration.pdf"

# Display the project locations being used
print("Project root :", PROJECT_ROOT.resolve())
print("Interim data :", INTERIM_DIR.resolve())
print("References   :", REFERENCE_DIR.resolve())

Project root : C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project
Interim data : C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\interim
References   : C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\raw\reference


In [4]:
# Store all files required for Notebook 02
required_files = [
    ELOG_FILE,
    IEBC_2013_FILE,
    IEBC_2017_DATA_FILE,
    IEBC_2017_EVAL_FILE,
    IEBC_2022_EVAL_FILE,
    GAZETTE_2022_FILE
]

# Confirm that every required file exists locally
for file in required_files:
    print(f"{'✓' if file.exists() else '✗'} {file.name}")

# Stop the notebook if any required file is missing
assert all(file.exists() for file in required_files), (
    "One or more required files are missing."
)

✓ elog_county_election_results.csv
✓ IEBC_2013_General_Election_Results.pdf
✓ IEBC_2017_Presidential_Declaration_Results.pdf
✓ IEBC_2017_Post_Election_Evaluation_Report.pdf
✓ IEBC_2022_Post_Election_Evaluation_Report.pdf
✓ Kenya_Gazette_2022_Presidential_Declaration.pdf


In [5]:
# Load the structured ELOG dataset created in Notebook 01
elog_df = pd.read_csv(ELOG_FILE)

# Confirm the expected dataset size
print("ELOG shape:", elog_df.shape)

# Confirm records by election
display(
    elog_df.groupby("election_id").size()
)

# Preview the source data
display(elog_df.head())

ELOG shape: (188, 16)


election_id
2013_GENERAL        47
2017_AUG_GENERAL    47
2017_OCT_FRESH      47
2022_GENERAL        47
dtype: int64

,election_id,election_year,election_date,election_type,county,registered_voters,valid_votes,turnout_pct,winner,winner_votes,runner_up,margin,source,source_file,valid_vote_rate_pct,turnout_difference_pp
0,2013_GENERAL,2013,2013-03-04,General Election,Mombasa,408747,269314,66.62,Raila Odinga,189985,Uhuru Kenyatta,125192,ELOG,ELOG_2013_Presidential_General_Election.pdf,65.887701,0.732299
1,2013_GENERAL,2013,2013-03-04,General Election,Kwale,174443,124626,72.00,Raila Odinga,101407,Uhuru Kenyatta,83774,ELOG,ELOG_2013_Presidential_General_Election.pdf,71.442248,0.557752
2,2013_GENERAL,2013,2013-03-04,General Election,Kilifi,336132,215792,64.91,Raila Odinga,182708,Uhuru Kenyatta,159322,ELOG,ELOG_2013_Presidential_General_Election.pdf,64.198589,0.711411
3,2013_GENERAL,2013,2013-03-04,General Election,Tana River,79454,64080,81.29,Raila Odinga,39666,Uhuru Kenyatta,17247,ELOG,ELOG_2013_Presidential_General_Election.pdf,80.650439,0.639561
4,2013_GENERAL,2013,2013-03-04,General Election,Lamu,52346,43534,84.38,Raila Odinga,22962,Uhuru Kenyatta,5285,ELOG,ELOG_2013_Presidential_General_Election.pdf,83.165858,1.214142


In [6]:
# Open the official 2013 IEBC election report
reader_2013 = PdfReader(IEBC_2013_FILE)

# Extract the pages containing the county presidential summary
text_2013 = " ".join(
    reader_2013.pages[page].extract_text() or ""
    for page in [5, 6, 7]
)

# Collapse repeated whitespace to make table rows easier to parse
text_2013 = re.sub(r"\s+", " ", text_2013)

# Preview a small section of the extracted table
print(text_2013[:1500])

4 1. Summary of Presidential Election Results for March 4th 2013 NO. COUNTY COUNTY POPULATION (CENSUS 2009) ESTIMATED VOTING POPULATION NO OF IDS ISSUED (As at 13-Nov 2012)* REG VOTERS (BIOMETRIC S) REG VOTERS (NON BIOMETRIC ) TOTAL REG VOTERS NUMBER THAT VOTED 04/03/2013 REG VOTERS MINUS VOTED 4/3/13 VOTER TURNOUT 1 MOMBASA 939,370 437,934 569,847 408,747 4,322 413,069 272,318 140,751 66% 2 KWALE 649,931 302,997 303,456 174,443 1,129 175,572 125,601 49,971 72% 3 KILIFI 1,109,735 517,358 537,316 336,132 278 336,410 218,174 118,236 65% 4 TANA RIVER 240,075 111,923 110,202 79,454 187 79,641 64,589 15,052 81% 5 LAMU 101,539 47,337 63,578 52,346 13 52,359 44,171 8,188 84% 6 TAITA TAVETA 284,657 132,707 187,243 113,862 327 114,189 92,356 21,833 81% COAST 3,325,307 1,550,256 1,771,642 1,164,984 6,256 1,171,240 817,209 354,031 70% 7 GARISSA 623,060 290,470 158,682 115,202 34 115,236 91,900 23,336 80% 8 WAJIR 661,941 308,596 145,517 118,091 154 118,245 100,239 18,006 85% 9 MANDERA 1,025,756 47

In [7]:
# Define the structure of the official 2013 county rows
pattern_2013 = re.compile(
    r"\b(?P<county_no>\d{1,2})\s+"
    r"(?P<county>[A-Z][A-Z\s\-/']+?)\s+"
    r"(?P<population>[\d,]+)\s+"
    r"(?P<estimated_voting_population>[\d,]+)\s+"
    r"(?P<ids_issued>[\d,]+)\s+"
    r"(?P<registered_biometric>[\d,]+)\s+"
    r"(?P<registered_non_biometric>[\d,]+)\s+"
    r"(?P<registered_total>[\d,]+)\s+"
    r"(?P<number_voted>[\d,]+)\s+"
    r"(?P<registered_minus_voted>[\d,]+)\s+"
    r"(?P<turnout_pct>\d+)%"
)

# Extract all county rows from the official table
rows_2013 = []

for match in pattern_2013.finditer(text_2013):

    row = match.groupdict()

    rows_2013.append({
        "county_no": int(row["county_no"]),
        "county_raw": row["county"].strip(),
        "registered_biometric": int(row["registered_biometric"].replace(",", "")),
        "registered_non_biometric": int(row["registered_non_biometric"].replace(",", "")),
        "registered_total": int(row["registered_total"].replace(",", "")),
        "number_voted": int(row["number_voted"].replace(",", "")),
        "official_turnout_pct": float(row["turnout_pct"])
    })

# Convert the extracted records into a DataFrame
iebc_2013 = pd.DataFrame(rows_2013)

print("2013 counties extracted:", len(iebc_2013))

display(iebc_2013.head())

2013 counties extracted: 47


,county_no,county_raw,registered_biometric,registered_non_biometric,registered_total,number_voted,official_turnout_pct
0,1,MOMBASA,408747,4322,413069,272318,66.0
1,2,KWALE,174443,1129,175572,125601,72.0
2,3,KILIFI,336132,278,336410,218174,65.0
3,4,TANA RIVER,79454,187,79641,64589,81.0
4,5,LAMU,52346,13,52359,44171,84.0


In [8]:
# Confirm that all 47 counties were successfully extracted
assert len(iebc_2013) == 47, (
    f"Expected 47 counties, found {len(iebc_2013)}."
)

# Check for duplicate county numbers
assert iebc_2013["county_no"].nunique() == 47

# Inspect official national totals calculated from county rows
print("Biometric registered:",
      iebc_2013["registered_biometric"].sum())

print("Non-biometric registered:",
      iebc_2013["registered_non_biometric"].sum())

print("Total registered:",
      iebc_2013["registered_total"].sum())

print("Number voted:",
      iebc_2013["number_voted"].sum())

Biometric registered: 14349908
Non-biometric registered: 36236
Total registered: 14386144
Number voted: 12327700


In [9]:
# Create a function to standardise county names across sources
def standardise_county_name(name):

    # Convert to uppercase and remove inconsistent punctuation
    name = str(name).upper().strip()
    name = name.replace("’", "'")
    name = name.replace("-", " ")
    name = name.replace("/", " ")
    name = name.replace("'", "")

    # Remove repeated whitespace
    name = re.sub(r"\s+", " ", name)

    # Correct known source-specific naming differences
    aliases = {
        "THARAKA": "THARAKA NITHI",
        "THARAKA NITHI": "THARAKA NITHI",
        "MURANGA": "MURANGA",
        "ELGEYO MARAKWET": "ELGEYO MARAKWET",
        "HOMABAY": "HOMA BAY",
        "HOMA BAY": "HOMA BAY",
        "NAIROBI": "NAIROBI CITY",
        "NAIROBI CITY": "NAIROBI CITY",
        "LAI KI PI A": "LAIKIPIA",
        "KIRI NY AG A": "KIRINYAGA"
    }

    return aliases.get(name, name)


# Standardise county names in the official 2013 dataset
iebc_2013["county_key"] = (
    iebc_2013["county_raw"]
    .apply(standardise_county_name)
)

# Standardise county names in the ELOG dataset
elog_df["county_key"] = (
    elog_df["county"]
    .apply(standardise_county_name)
)

# Confirm the resulting official county names
display(
    iebc_2013[["county_raw", "county_key"]]
)

,county_raw,county_key
0,MOMBASA,MOMBASA
1,KWALE,KWALE
2,KILIFI,KILIFI
3,TANA RIVER,TANA RIVER
4,LAMU,LAMU
5,TAITA TAVETA,TAITA TAVETA
6,GARISSA,GARISSA
7,WAJIR,WAJIR
8,MANDERA,MANDERA
9,MARSABIT,MARSABIT


In [10]:
# Select the 2013 ELOG records
elog_2013 = (
    elog_df[elog_df["election_id"] == "2013_GENERAL"]
    .copy()
)

# Merge ELOG and official IEBC data by standardised county
recon_2013 = elog_2013.merge(
    iebc_2013,
    on="county_key",
    how="outer",
    indicator=True
)

# Calculate differences against both official registration definitions
recon_2013["diff_vs_biometric"] = (
    recon_2013["registered_voters"]
    -
    recon_2013["registered_biometric"]
)

recon_2013["diff_vs_total_register"] = (
    recon_2013["registered_voters"]
    -
    recon_2013["registered_total"]
)

# Display the key reconciliation fields
display(
    recon_2013[
        [
            "county",
            "registered_voters",
            "registered_biometric",
            "registered_non_biometric",
            "registered_total",
            "diff_vs_biometric",
            "diff_vs_total_register"
        ]
    ]
)

,county,registered_voters,registered_biometric,registered_non_biometric,registered_total,diff_vs_biometric,diff_vs_total_register
0,Mombasa,408747,408747,4322,413069,0,-4322
1,Kwale,174443,174443,1129,175572,0,-1129
2,Kilifi,336132,336132,278,336410,0,-278
3,Tana River,79454,79454,187,79641,0,-187
4,Lamu,52346,52346,13,52359,0,-13
5,Taita Taveta,113862,113862,327,114189,0,-327
6,Garissa,115202,115202,34,115236,0,-34
7,Wajir,118091,118091,154,118245,0,-154
8,Mandera,120768,120768,155,120923,0,-155
9,Marsabit,104615,104615,644,105259,0,-644


In [11]:
# Summarise how closely ELOG matches the official biometric register
recon_2013["biometric_match"] = (
    recon_2013["diff_vs_biometric"] == 0
)

# Count exact and non-exact matches
exact_matches = recon_2013["biometric_match"].sum()
non_matches = (~recon_2013["biometric_match"]).sum()

print("Exact biometric matches :", exact_matches)
print("Non-exact matches       :", non_matches)

# Display only counties where ELOG differs from the official biometric figure
display(
    recon_2013.loc[
        ~recon_2013["biometric_match"],
        [
            "county",
            "registered_voters",
            "registered_biometric",
            "diff_vs_biometric",
            "registered_total"
        ]
    ]
)

Exact biometric matches : 42
Non-exact matches       : 5


,county,registered_voters,registered_biometric,diff_vs_biometric,registered_total
18,Nyeri,356380,356381,-1,357059
21,Kiambu,861828,861829,-1,863199
26,Uasin Gishu,330618,330630,-12,332177
31,Nakuru,695319,695318,1,696594
32,Narok,262739,262738,1,263365


In [12]:
# Classify differences between ELOG and the official biometric register
def classify_2013_registration(row):

    difference = abs(row["diff_vs_biometric"])

    # Exact agreement between sources
    if difference == 0:
        return "MATCH"

    # Very small differences requiring source review
    elif difference <= 12:
        return "MINOR_SOURCE_DIFFERENCE"

    # Larger differences require investigation before use
    else:
        return "REVIEW"


# Apply reconciliation status
recon_2013["registration_status"] = (
    recon_2013.apply(
        classify_2013_registration,
        axis=1
    )
)

# Review the number of records in each status
display(
    recon_2013["registration_status"]
    .value_counts()
)

registration_status
MATCH                      42
MINOR_SOURCE_DIFFERENCE     5
Name: count, dtype: int64

In [13]:
# Open the official IEBC 2017 elections data report
reader_2017 = PdfReader(IEBC_2017_DATA_FILE)

# Extract the two pages containing the August presidential county table
august_pages = [
    reader_2017.pages[17].extract_text() or "",
    reader_2017.pages[18].extract_text() or ""
]

# Store parsed county records
rows_aug_2017 = []

# Process each page separately
for page_text in august_pages:

    # Split the extracted page text into cleaned lines
    lines = [
        " ".join(line.split())
        for line in page_text.splitlines()
        if line.strip()
    ]

    # Loop through lines using an index so continuation lines can be joined
    i = 0

    while i < len(lines):

        line = lines[i]

        # Look for rows beginning with an official county code
        match = re.match(
            r"^(?P<county_code>\d{2})\s+(?P<body>.+)$",
            line
        )

        if not match:
            i += 1
            continue

        county_code = int(match.group("county_code"))

        # Keep only Kenya's 47 counties
        if not 1 <= county_code <= 47:
            i += 1
            continue

        body = match.group("body")

        # Extract numeric values from the current line
        values = re.findall(r"\d[\d,]*", body)

        # If the row is incomplete, append the next line
        # This handles wrapped records such as ELGEYO/MARAKWET
        while len(values) < 3 and i + 1 < len(lines):

            i += 1

            continuation = lines[i]

            # Append the continuation text to the current row
            body = body + " " + continuation

            # Re-extract numeric values after joining the lines
            values = re.findall(r"\d[\d,]*", body)

        # A valid county record needs at least:
        # registered voters, valid votes and rejected ballots
        if len(values) < 3:
            print(
                f"Unable to parse county {county_code}:",
                body
            )

            i += 1
            continue

        # Store only the fields required for reconciliation
        rows_aug_2017.append({
            "county_code": county_code,
            "registered_voters_official": int(
                values[0].replace(",", "")
            ),
            "valid_votes_official": int(
                values[-2].replace(",", "")
            ),
            "rejected_ballots_official": int(
                values[-1].replace(",", "")
            )
        })

        i += 1


# Convert the extracted records into a DataFrame
iebc_aug_2017 = pd.DataFrame(rows_aug_2017)

# Sort records using official county codes
iebc_aug_2017 = (
    iebc_aug_2017
    .sort_values("county_code")
    .reset_index(drop=True)
)

# Display extraction result
print(
    "August 2017 counties extracted:",
    len(iebc_aug_2017)
)

display(iebc_aug_2017.head())

August 2017 counties extracted: 47


,county_code,registered_voters_official,valid_votes_official,rejected_ballots_official
0,1,580644,342337,2838
1,2,281102,185204,1316
2,3,508425,327831,2181
3,4,118338,86920,711
4,5,69793,49214,894


In [14]:
# Define Kenya's county code sequence used by IEBC
COUNTY_CODE_MAP = {
    1: "MOMBASA",
    2: "KWALE",
    3: "KILIFI",
    4: "TANA RIVER",
    5: "LAMU",
    6: "TAITA TAVETA",
    7: "GARISSA",
    8: "WAJIR",
    9: "MANDERA",
    10: "MARSABIT",
    11: "ISIOLO",
    12: "MERU",
    13: "THARAKA NITHI",
    14: "EMBU",
    15: "KITUI",
    16: "MACHAKOS",
    17: "MAKUENI",
    18: "NYANDARUA",
    19: "NYERI",
    20: "KIRINYAGA",
    21: "MURANGA",
    22: "KIAMBU",
    23: "TURKANA",
    24: "WEST POKOT",
    25: "SAMBURU",
    26: "TRANS NZOIA",
    27: "UASIN GISHU",
    28: "ELGEYO MARAKWET",
    29: "NANDI",
    30: "BARINGO",
    31: "LAIKIPIA",
    32: "NAKURU",
    33: "NAROK",
    34: "KAJIADO",
    35: "KERICHO",
    36: "BOMET",
    37: "KAKAMEGA",
    38: "VIHIGA",
    39: "BUNGOMA",
    40: "BUSIA",
    41: "SIAYA",
    42: "KISUMU",
    43: "HOMA BAY",
    44: "MIGORI",
    45: "KISII",
    46: "NYAMIRA",
    47: "NAIROBI CITY"
}

# Add standard county names using official county codes
iebc_aug_2017["county_key"] = (
    iebc_aug_2017["county_code"]
    .map(COUNTY_CODE_MAP)
)

In [15]:
# Confirm that all 47 county records were extracted
assert len(iebc_aug_2017) == 47

# Confirm that each county code appears once
assert iebc_aug_2017["county_code"].nunique() == 47

# Calculate national totals from county rows
print(
    "County registered total:",
    iebc_aug_2017["registered_voters_official"].sum()
)

print(
    "County valid-vote total:",
    iebc_aug_2017["valid_votes_official"].sum()
)

print(
    "County rejected-ballot total:",
    iebc_aug_2017["rejected_ballots_official"].sum()
)

County registered total: 19607064
County valid-vote total: 15142702
County rejected-ballot total: 81685


In [16]:
# Select the August 2017 ELOG records
elog_aug_2017 = (
    elog_df[
        elog_df["election_id"] == "2017_AUG_GENERAL"
    ]
    .copy()
)

# Merge ELOG and official IEBC records
recon_aug_2017 = elog_aug_2017.merge(
    iebc_aug_2017,
    on="county_key",
    how="outer",
    indicator=True
)

# Calculate registration differences
recon_aug_2017["registered_difference"] = (
    recon_aug_2017["registered_voters"]
    -
    recon_aug_2017["registered_voters_official"]
)

# Calculate valid-vote differences
recon_aug_2017["valid_votes_difference"] = (
    recon_aug_2017["valid_votes"]
    -
    recon_aug_2017["valid_votes_official"]
)

# Calculate official votes cast using valid plus rejected ballots
recon_aug_2017["votes_cast_official"] = (
    recon_aug_2017["valid_votes_official"]
    +
    recon_aug_2017["rejected_ballots_official"]
)

# Calculate turnout from official vote totals
recon_aug_2017["turnout_calculated_pct"] = (
    recon_aug_2017["votes_cast_official"]
    /
    recon_aug_2017["registered_voters_official"]
) * 100

# Compare ELOG's turnout against the official-derived turnout
recon_aug_2017["turnout_difference_pp"] = (
    recon_aug_2017["turnout_pct"]
    -
    recon_aug_2017["turnout_calculated_pct"]
)

display(
    recon_aug_2017[
        [
            "county",
            "registered_voters",
            "registered_voters_official",
            "registered_difference",
            "valid_votes",
            "valid_votes_official",
            "valid_votes_difference",
            "rejected_ballots_official",
            "votes_cast_official",
            "turnout_pct",
            "turnout_calculated_pct",
            "turnout_difference_pp"
        ]
    ]
)

,county,registered_voters,registered_voters_official,registered_difference,valid_votes,valid_votes_official,valid_votes_difference,rejected_ballots_official,votes_cast_official,turnout_pct,turnout_calculated_pct,turnout_difference_pp
0,Mombasa,580644,580644,0,342337,342337,0,2838,345175,58.95,59.446924,-0.496924
1,Kwale,281102,281102,0,185204,185204,0,1316,186520,65.75,66.353139,-0.603139
2,Kilifi,508425,508425,0,327831,327831,0,2181,330012,64.51,64.908689,-0.398689
3,Tana River,118338,118338,0,86920,86920,0,711,87631,73.90,74.051446,-0.151446
4,Lamu,69793,69793,0,49214,49214,0,894,50108,70.96,71.795166,-0.835166
5,Taita Taveta,155794,155794,0,112683,112683,0,736,113419,72.27,72.800621,-0.530621
6,Garissa,163350,163350,0,113236,113236,0,883,114119,69.49,69.861647,-0.371647
7,Wajir,162912,162912,0,118688,118688,0,656,119344,73.37,73.256728,0.113272
8,Mandera,175650,175650,0,135681,135681,0,198,135879,77.80,77.357814,0.442186
9,Marsabit,141730,141730,0,110839,110839,0,299,111138,78.20,78.415297,-0.215297


In [17]:
# Extract October 2017 fresh-election table pages
october_pages = [
    reader_2017.pages[page].extract_text() or ""
    for page in [21, 22, 23, 24, 25]
]

# Store extracted county records
rows_oct_2017 = []

for page_text in october_pages:

    for line in page_text.splitlines():

        # Normalise repeated whitespace
        line = " ".join(line.split())

        # Only process county total rows
        if "COUNTY TOTAL" not in line:
            continue

        # Ignore diaspora because our main geography is the 47 counties
        if line.startswith("DIASPORA"):
            continue

        # Extract numeric fields
        values = re.findall(
            r"\d[\d,]*",
            line
        )

        # Extract the county name appearing before COUNTY TOTAL
        county_raw = line.split("COUNTY TOTAL")[0].strip()

        # Standardise the county name
        county_key = standardise_county_name(county_raw)

        # Some counties recorded zero valid votes in the fresh election
        if len(values) == 1:

            registered = int(values[0].replace(",", ""))
            valid_votes = 0
            rejected_ballots = 0

        elif len(values) == 11:

            registered = int(values[0].replace(",", ""))
            valid_votes = int(values[-2].replace(",", ""))
            rejected_ballots = int(values[-1].replace(",", ""))

        else:
            print("Review row:", line)
            continue

        rows_oct_2017.append({
            "county_key": county_key,
            "registered_voters_official": registered,
            "valid_votes_official": valid_votes,
            "rejected_ballots_official": rejected_ballots
        })

# Convert to DataFrame
iebc_oct_2017 = pd.DataFrame(rows_oct_2017)

print("October 2017 counties extracted:", len(iebc_oct_2017))

display(
    iebc_oct_2017.sort_values("valid_votes_official").head(10)
)

October 2017 counties extracted: 47


,county_key,registered_voters_official,valid_votes_official,rejected_ballots_official
40,SIAYA,457957,0,0
42,HOMA BAY,476932,0,0
41,KISUMU,539597,899,5
37,VIHIGA,272415,10584,248
16,MAKUENI,423434,19409,224
10,ISIOLO,75355,20131,114
4,LAMU,69793,23073,376
39,BUSIA,351087,23554,250
8,MANDERA,175650,24834,30
5,TAITA TAVETA,155926,25877,214


In [18]:
# Select ELOG October 2017 records
elog_oct_2017 = (
    elog_df[
        elog_df["election_id"] == "2017_OCT_FRESH"
    ]
    .copy()
)

# Merge ELOG with official fresh-election records
recon_oct_2017 = elog_oct_2017.merge(
    iebc_oct_2017,
    on="county_key",
    how="outer",
    indicator=True
)

# Calculate differences in key measures
recon_oct_2017["registered_difference"] = (
    recon_oct_2017["registered_voters"]
    -
    recon_oct_2017["registered_voters_official"]
)

recon_oct_2017["valid_votes_difference"] = (
    recon_oct_2017["valid_votes"]
    -
    recon_oct_2017["valid_votes_official"]
)

# Calculate official votes cast
recon_oct_2017["votes_cast_official"] = (
    recon_oct_2017["valid_votes_official"]
    +
    recon_oct_2017["rejected_ballots_official"]
)

# Calculate turnout from official fields
recon_oct_2017["turnout_calculated_pct"] = (
    recon_oct_2017["votes_cast_official"]
    /
    recon_oct_2017["registered_voters_official"]
) * 100

display(
    recon_oct_2017[
        [
            "county",
            "registered_voters",
            "registered_voters_official",
            "valid_votes",
            "valid_votes_official",
            "rejected_ballots_official",
            "turnout_pct",
            "turnout_calculated_pct"
        ]
    ]
    .sort_values("turnout_calculated_pct")
)

,county,registered_voters,registered_voters_official,valid_votes,valid_votes_official,rejected_ballots_official,turnout_pct,turnout_calculated_pct
42,Homa Bay,476932,476932,0,0,0,0.00,0.000000
40,Siaya,457957,457957,0,0,0,0.00,0.000000
41,Kisumu,539597,539597,899,899,5,0.17,0.167532
37,Vihiga,272415,272415,10584,10584,248,3.98,3.976286
16,Makueni,423434,423434,19409,19409,224,4.64,4.636614
36,Kakamega,743929,743929,43936,43936,755,6.01,6.007428
39,Busia,351087,351087,23554,23554,250,6.78,6.780086
2,Kilifi,508266,508266,42122,42122,441,8.37,8.374158
43,Migori,388696,388696,33390,33390,184,8.64,8.637599
14,Kitui,474563,474563,50370,50370,425,10.70,10.703531


In [19]:
# Create official national benchmarks from the 2022 IEBC evaluation report
official_2022 = {
    "election_id": "2022_GENERAL",
    "registered_voters_official": 22_120_458,
    "turnout_pct_official": 64.77
}

# Select 2022 ELOG county records
elog_2022 = (
    elog_df[
        elog_df["election_id"] == "2022_GENERAL"
    ]
    .copy()
)

# Aggregate the ELOG county registered-voter figures
elog_2022_registered = elog_2022["registered_voters"].sum()

# Compare the aggregated ELOG figure with the official national register
print(
    "ELOG county registered total :",
    f"{elog_2022_registered:,}"
)

print(
    "Official national register   :",
    f"{official_2022['registered_voters_official']:,}"
)

print(
    "Difference                   :",
    f"{elog_2022_registered - official_2022['registered_voters_official']:,}"
)

ELOG county registered total : 22,110,015
Official national register   : 22,120,458
Difference                   : -10,443


In [20]:
# Create a high-level reconciliation summary
reconciliation_summary = pd.DataFrame([
    {
        "election_id": "2013_GENERAL",
        "county_records_expected": 47,
        "county_records_official": len(iebc_2013),
        "official_source": "IEBC 2013 General Election Data",
        "reconciliation_level": "County"
    },
    {
        "election_id": "2017_AUG_GENERAL",
        "county_records_expected": 47,
        "county_records_official": len(iebc_aug_2017),
        "official_source": "IEBC Data Report of 2017 Elections",
        "reconciliation_level": "County"
    },
    {
        "election_id": "2017_OCT_FRESH",
        "county_records_expected": 47,
        "county_records_official": len(iebc_oct_2017),
        "official_source": "IEBC Data Report of 2017 Elections",
        "reconciliation_level": "County"
    },
    {
        "election_id": "2022_GENERAL",
        "county_records_expected": 47,
        "county_records_official": None,
        "official_source": "IEBC 2022 Post Election Evaluation",
        "reconciliation_level": "National only"
    }
])

display(reconciliation_summary)

,election_id,county_records_expected,county_records_official,official_source,reconciliation_level
0,2013_GENERAL,47,47.0,IEBC 2013 General Election Data,County
1,2017_AUG_GENERAL,47,47.0,IEBC Data Report of 2017 Elections,County
2,2017_OCT_FRESH,47,47.0,IEBC Data Report of 2017 Elections,County
3,2022_GENERAL,47,NaN,IEBC 2022 Post Election Evaluation,National only


In [21]:
# Define output locations for official extracts and reconciliation results
IEBC_2013_OUTPUT = INTERIM_DIR / "iebc_2013_county_official.csv"
IEBC_2017_AUG_OUTPUT = INTERIM_DIR / "iebc_2017_august_county_official.csv"
IEBC_2017_OCT_OUTPUT = INTERIM_DIR / "iebc_2017_october_county_official.csv"

RECON_2013_OUTPUT = DOCS_DIR / "reconciliation_2013.csv"
RECON_2017_AUG_OUTPUT = DOCS_DIR / "reconciliation_2017_august.csv"
RECON_2017_OCT_OUTPUT = DOCS_DIR / "reconciliation_2017_october.csv"

SUMMARY_OUTPUT = DOCS_DIR / "notebook02_reconciliation_summary.csv"

# Save official structured extracts
iebc_2013.to_csv(
    IEBC_2013_OUTPUT,
    index=False
)

iebc_aug_2017.to_csv(
    IEBC_2017_AUG_OUTPUT,
    index=False
)

iebc_oct_2017.to_csv(
    IEBC_2017_OCT_OUTPUT,
    index=False
)

# Save detailed reconciliation outputs
recon_2013.to_csv(
    RECON_2013_OUTPUT,
    index=False
)

recon_aug_2017.to_csv(
    RECON_2017_AUG_OUTPUT,
    index=False
)

recon_oct_2017.to_csv(
    RECON_2017_OCT_OUTPUT,
    index=False
)

# Save Notebook 02 summary
reconciliation_summary.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

print("Notebook 02 outputs saved successfully.")

Notebook 02 outputs saved successfully.
